# Анализ

In [ ]:
import os
import glob
from collections import defaultdict
import numpy as np

[{'id': 0, 'name': 'football-objects', 'supercategory': 'none'},
 {'id': 1, 'name': 'ball', 'supercategory': 'football-objects'},
 {'id': 2, 'name': 'coach', 'supercategory': 'football-objects'},
 {'id': 3, 'name': 'goalkeeper', 'supercategory': 'football-objects'},
 {'id': 4, 'name': 'player', 'supercategory': 'football-objects'},
 {'id': 5, 'name': 'referee', 'supercategory': 'football-objects'}]

labels = {1: 'ball', 2: 'coach', 3: 'goalkeeper', 4: 'player', 5: 'referee'}

def process_labels_folder(labels_dir):
    list_w, list_h = [], []
    class_total_area = defaultdict(float)
    class_total_w = defaultdict(float)
    class_total_h = defaultdict(float)
    class_count = defaultdict(int)

    # Поиск всех .txt файлов в папке labels
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))

    for file_path in label_files:
        with open(file_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue

            try:
                cls = int(parts[0])
                w = float(parts[3])
                h = float(parts[4])
                area = w * h

                class_total_area[cls] += area
                class_total_w[cls] += w
                class_total_h[cls] += h
                list_w.append(w)
                list_h.append(h)
                class_count[cls] += 1

            except (ValueError, IndexError):
                print(f"Warning: некорректная строка в {file_path}: {line}")
                continue

    print("Статистика ширины и высота боксов по классам:")
    for cls in sorted(class_total_area.keys()):
        avg_area = class_total_area[cls] / class_count[cls]
        avg_w = class_total_w[cls] / class_count[cls]
        avg_h = class_total_h[cls] / class_count[cls]
        print(f"Класс {labels[cls]:<11} avg_w = {avg_w:.4f}, avg_h = {avg_h:.4f}, (всего боксов: {class_count[cls]})")

    np_list_w = np.array(list_w)
    np_list_h = np.array(list_h)

    print("\nОбщая статистика ширины и высота боксов:")
    print(f"min_w = {np_list_w.min()}, min_h = {np_list_h.min()}")
    print(f"max_w = {np_list_w.max()}, max_h = {np_list_h.max()}")
    print(f"quantile_75 = {np.quantile(np_list_h, 0.75):.4f}")
    print(f"quantile_85 = {np.quantile(np_list_h, 0.85):.4f}")
    print(f"quantile_95 = {np.quantile(np_list_h, 0.95):.4f}")
    print(f"quantile_99 = {np.quantile(np_list_h, 0.99):.4f}")

labels_dir = "D:\model_descriptor\ML_MAGA\project\coco_yolo\labels\\train"  # ← измените, если папка называется иначе или лежит в другом месте
if not os.path.isdir(labels_dir):
    print(f"Ошибка: папка '{labels_dir}' не найдена.")
    exit(1)
process_labels_folder(labels_dir)

Статистика ширины и высота боксов по классам:
Класс ball        avg_w = 0.0086, avg_h = 0.0156, (всего боксов: 591)
Класс coach       avg_w = 0.0245, avg_h = 0.0834, (всего боксов: 164)
Класс goalkeeper  avg_w = 0.0236, avg_h = 0.0785, (всего боксов: 219)
Класс player      avg_w = 0.0231, avg_h = 0.0850, (всего боксов: 9901)
Класс referee     avg_w = 0.0218, avg_h = 0.0766, (всего боксов: 1088)

Общая статистика ширины и высота боксов:
min_w = 0.00038, min_h = 0.000556
max_w = 0.087818, max_h = 0.351852
quantile_75 = 0.0955
quantile_85 = 0.1033
quantile_95 = 0.1188
quantile_99 = 0.1415


# Обучение

In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def load_yolo_annotations(img_path, label_path):
    """
    Загружает изображение и аннотации в формате YOLO.
    Возвращает: image [3, H, W], targets [M, 5] = [cls, x, y, w, h] (нормализованные)
    !!!ВОЗВРАЩАЕТ классы с индексами от 0, даже если на вход идут начиная с 1!!!
    """
    # Загрузка изображения
    img = Image.open(img_path).convert('RGB')
    w, h = img.size

    # Загрузка аннотаций
    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        targets = torch.empty(0, 5)
    else:
        boxes = []
        with open(label_path) as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) < 5: continue
                cls_id, x, y, w, h = parts[:5]
                boxes.append([cls_id-1, x, y, w, h])
        targets = torch.tensor(boxes) if boxes else torch.empty(0, 5)

        if (targets[:, 0].min == 1):
            targets[:, 0] -= 1

    return img, targets


class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir, imgsz=640, augment=True):
        self.img_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        self.label_paths = [p.replace('/images/', '/labels/').replace('.jpg', '.txt').replace('.png', '.txt') for p in self.img_paths]
        self.imgsz = imgsz
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img, targets = load_yolo_annotations(self.img_paths[idx], self.label_paths[idx])
        
        # Здесь можно добавить аугментации (mosaic, hsv, flip)
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((640, 640)),
        ])
        img = transform(img)
    
        return img, targets

In [3]:
import torch
import torch.optim as optim
from tqdm import tqdm
import time
from collections import defaultdict

@torch.no_grad()
def class_weight(targets, num_classes, device):
    class_count = defaultdict(int)
    for t in targets:
        if t.numel() == 0:
            continue
        class_ids = t[:, 0].long().cpu().numpy()
        for cls in class_ids:
            class_count[cls] += 1
    total_samples = sum(class_count.values())
    
    cls_weight = []
    for cls in range(num_classes):  # например, 5 классов
        pos_count = class_count.get(cls, 0)
        neg_count = total_samples - pos_count
        if pos_count == 0:
            weight = 1.0
        else:
            weight = neg_count / pos_count
        cls_weight.append(weight)

    cls_weight = torch.tensor(cls_weight, device=device)

    return cls_weight

def train_one_epoch(model, dataloader, optimizer, scaler, loss_fn, device, epoch, imgsz=640):
    model.train()
    total_loss = 0.0
    loss_logs = {"loss_box": 0.0, "loss_obj": 0.0, "loss_cls": 0.0, "num_pos": 0}

    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch}")
    for batch_idx, (images, targets) in pbar:
        images = images.to(device)
        targets = [t.to(device) for t in targets]

        cls_weight = class_weight(targets, model.nc_external, device)

        with torch.amp.autocast(device.type):
            optimizer.zero_grad()
            pred = model(images)
        
            loss, logs = loss_fn(
                pred, targets,
                pos_weight = cls_weight,
                num_classes=model.nc_external,
                imgsz=imgsz,
                device=device,
                weight_box=7.5,   # CIoU/GIoU loss обычно сильнее
                weight_obj=1.0,
                weight_cls=0.5    # классы — менее критичны, чем bbox
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # optimizer.step()

        total_loss += loss.item()
        for k in loss_logs:
            loss_logs[k] += logs[k]

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "obj": f"{logs['loss_obj']:.4f}",
            "cls": f"{logs['loss_cls']:.4f}",
            "box": f"{logs['loss_box']:.4f}",
            "pos": f"{logs['num_pos']}"
        })

    avg_loss = total_loss / len(dataloader)
    for k in loss_logs:
        loss_logs[k] /= len(dataloader)

    return avg_loss, loss_logs

@torch.no_grad()
def validate_one_epoch(model, dataloader, loss_fn, device, imgsz=640):
    model.eval()
    total_loss = 0.0
    loss_logs = {"loss_box": 0.0, "loss_obj": 0.0, "loss_cls": 0.0, "num_pos": 0}

    pbar = tqdm(dataloader, desc="Validate", leave=False)
    for images, targets in pbar:
        images = images.to(device)
        targets = [t.to(device) for t in targets]
        cls_weight = class_weight(targets, model.nc_external, device)

        pred = model(images)

        # Защита от inf/nan и на валидации (на случай, если модель "испортилась")
        if torch.isnan(pred).any() or torch.isinf(pred).any():
            print("⚠️  Warning: pred has NaN/Inf during validation — skipping batch")
            continue

        loss, logs = loss_fn(
            pred, targets,
            pos_weight=cls_weight,
            num_classes=model.nc_external,
            imgsz=imgsz,
            device=device,
            weight_box=7.5,
            weight_obj=1.0,
            weight_cls=0.5
        )

        total_loss += loss.item()
        for k in loss_logs:
            loss_logs[k] += logs[k]

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0.0
    for k in loss_logs:
        loss_logs[k] /= len(dataloader) if len(dataloader) > 0 else 1

    return avg_loss, loss_logs

In [4]:
from detection_loss import detection_loss
from yolo_model import create_yolo_model
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print("Device:", device)

version = 5
imgsz = 640
batch_size = 16
num_epochs = 100
num_classes = 5  # ball, coach, goalkeeper, player, referee

model = create_yolo_model(num_classes, imgsz=640)
model.to(device)

# ИЗМЕНИТЬ ДЛЯ ИСПОЛЬЗОВАНИЯ
PROJECT_PATH =  "D:/model_descriptor/ML_MAGA/project/"
GENERAL_PATH = PROJECT_PATH + "coco_yolo/"
TEST_PATH =    PROJECT_PATH + "dataset/test/"
WEIGHTS_PATH = PROJECT_PATH + "model/weights/"

train_dataset = YOLODataset(GENERAL_PATH + "images/train", GENERAL_PATH + "labels/train", imgsz=imgsz)
val_dataset = YOLODataset(GENERAL_PATH + "images/val", GENERAL_PATH + "labels/val", imgsz=imgsz, augment=False)

# collate_fn для списка targets
def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images, 0)
    return images, list(targets)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                            num_workers=0, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=True, collate_fn=collate_fn)

# 0.01 обычно для бОльших батчей
optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.937, weight_decay=0.0005, nesterov=True)
scaler = torch.amp.GradScaler()

# разогрев для первых эпох, чтобы градиенты не улетали, далее плавное сниэение через cos^2
def lr_lambda(epoch):
    if epoch < 3:
        return (epoch + 1) / 3
    return 0.5 * (1 + math.cos(math.pi * (epoch - 3) / (num_epochs - 3)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

min_val_loss = 0
for epoch in range(num_epochs):
    train_loss, train_logs = train_one_epoch(model, train_loader, optimizer, scaler, detection_loss, device, epoch, imgsz)
    val_loss, val_logs = validate_one_epoch(model, val_loader, detection_loss, device, imgsz)
    scheduler.step()

    # Обновлённый tqdm-статус с валидацией
    tqdm.write(
        f"Epoch {epoch} | "
        f"Train: loss={train_loss:.4f} (box={train_logs['loss_box']:.4f}, obj={train_logs['loss_obj']:.4f}, cls={train_logs['loss_cls']:.4f}) | "
        f"Val: loss={val_loss:.4f} (box={val_logs['loss_box']:.4f}, obj={val_logs['loss_obj']:.4f}, cls={val_logs['loss_cls']:.4f}) | "
        f"pos: {train_logs['num_pos']}/{val_logs['num_pos']}"
    )

    # Сохранение
    if val_loss < min_val_loss or epoch == 0:
        min_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        },  WEIGHTS_PATH + f"yolo_best_val_v{version}.pt")

model.eval()
model.apply(lambda m: hasattr(m, 'reparameterize') and m.reparameterize())
torch.save(model.state_dict(), WEIGHTS_PATH + f"yolo_final_rep_v{version}.pt")
print("✅ Model reparameterized and saved for inference!")

Device: cuda


Epoch 0: 100%|██████████| 40/40 [00:41<00:00,  1.04s/it, loss=4.2098, obj=0.2018, cls=1.3719, box=0.4429, pos=2996]


Epoch 0 | Train: loss=5.8735 (box=0.5052, obj=0.4347, cls=3.2990) | Val: loss=5.6323 (box=0.4793, obj=0.2724, cls=3.5302) | pos: 2852.3/2717.1


Epoch 1: 100%|██████████| 40/40 [00:31<00:00,  1.26it/s, loss=3.2845, obj=0.0479, cls=0.3202, box=0.4102, pos=3124]


Epoch 1 | Train: loss=4.0043 (box=0.4566, obj=0.0869, cls=0.9864) | Val: loss=3.6337 (box=0.4146, obj=0.0513, cls=0.9456) | pos: 2858.475/2719.5


...

Epoch 97 | Train: loss=1.4090 (box=0.1850, obj=0.0121, cls=0.0191) | Val: loss=1.9401 (box=0.2344, obj=0.0116, cls=0.3404) | pos: 2963.9/2811.1


Epoch 98: 100%|██████████| 40/40 [00:43<00:00,  1.08s/it, loss=1.3783, obj=0.0118, cls=0.0115, box=0.1814, pos=2821]


Epoch 98 | Train: loss=1.4118 (box=0.1852, obj=0.0121, cls=0.0208) | Val: loss=1.9379 (box=0.2344, obj=0.0116, cls=0.3366) | pos: 2964.0/2810.7


Epoch 99: 100%|██████████| 40/40 [00:41<00:00,  1.05s/it, loss=1.3373, obj=0.0125, cls=0.0129, box=0.1758, pos=3130]


Epoch 99 | Train: loss=1.4124 (box=0.1854, obj=0.0121, cls=0.0196) | Val: loss=1.9453 (box=0.2345, obj=0.0116, cls=0.3504) | pos: 2963.475/2810.8
✅ Model reparameterized and saved for inference!


In [6]:
model.eval()
# model.load_state_dict(torch.load(WEIGHTS_PATH + f"yolo_best_val_v{version}.pt"))
model.apply(lambda m: hasattr(m, 'reparameterize') and m.reparameterize())
torch.save(model.state_dict(), WEIGHTS_PATH + f"yolo_final_rep_v{version}.pt")

In [ ]:
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from torchvision.ops import nms, box_convert


def predict_and_visualize_v2(
    model,
    image,
    class_names,
    device='cuda',
    imgsz=640,
    conf_thres=0.5,
    iou_thres=0.45,
    max_dets=300
):
    """
    Predict and visualize with simple resize (no letterbox), PIL-based loading.
    
    Args:
        model: trained model → [B, N, 9]
        image: str (path) or PIL.Image
        class_names: list or dict, e.g. ['ball', 'player', ...]
        device: 'cuda' or 'cpu'
        imgsz: resize to (imgsz, imgsz) — no letterbox
        conf_thres: obj × cls confidence threshold
        iou_thres: NMS IoU threshold
        max_dets: max detections after NMS
    
    Returns:
        PIL.Image with boxes and labels
    """
    model.eval()
    
    # 1. Загрузка и препроцессинг через PIL + transforms
    if isinstance(image, str):
        pil_img_orig = Image.open(image).convert("RGB")
    elif isinstance(image, Image.Image):
        pil_img_orig = image.convert("RGB")
    else:
        raise TypeError("image must be str or PIL.Image")
    
    # Transform: resize → tensor → normalize
    transform = transforms.Compose([
        transforms.Resize((imgsz, imgsz)),
        transforms.ToTensor(),
    ])
    x = transform(pil_img_orig).unsqueeze(0).to(device)  # [1, 3, 640, 640]

    # 2. Inference
    with torch.no_grad(), torch.amp.autocast(device):
        pred = model(x)  # [1, N, 9]

    # 3. Декодирование предиктов
    pred = pred[0]  # [N, 9]
    box_cxcywh = pred[:, :4]          # [N, 4]
    obj_logit = pred[:, 4]            # [N]
    cls_logits = pred[:, 5:]          # [N, C]
    
    obj_conf = obj_logit.sigmoid()
    cls_conf = cls_logits.sigmoid()
    class_conf, class_id = cls_conf.max(dim=1)
    conf = obj_conf * class_conf

    # 4. Фильтрация по confidence
    keep = conf > conf_thres
    if keep.sum() == 0:
        return pil_img_orig.copy()

    box_cxcywh = box_cxcywh[keep]
    conf = conf[keep]
    class_id = class_id[keep]

    # 5. Конвертация в xyxy (в координатах 640×640)
    box_xyxy = box_convert(box_cxcywh, in_fmt='cxcywh', out_fmt='xyxy')

    # 6. ⭐ Class-Separated NMS (важно!)
    # Группируем по классам и делаем NMS внутри каждого класса
    keep_nms = []
    for cls in torch.unique(class_id):
        cls_mask = class_id == cls
        cls_boxes = box_xyxy[cls_mask]
        cls_conf = conf[cls_mask]
        cls_keep = nms(cls_boxes, cls_conf, iou_threshold=iou_thres)
        keep_nms.append(torch.where(cls_mask)[0][cls_keep])
    if keep_nms:
        keep_nms = torch.cat(keep_nms)
        keep_nms = keep_nms[conf[keep_nms].argsort(descending=True)[:max_dets]]  # top-k by confidence runs\weights\yolo_final_rep_v5.pt
    else:
        keep_nms = torch.tensor([], dtype=torch.long)
    if len(keep_nms) == 0:
        return pil_img_orig.copy()
    box_xyxy = box_xyxy[keep_nms]
    conf = conf[keep_nms]
    class_id = class_id[keep_nms]

    # cls_boxes = box_xyxy
    # cls_conf = conf
    # cls_keep = nms(cls_boxes, cls_conf, iou_threshold=iou_thres)
    # cls_keep = cls_keep[:max_dets]

    # box_xyxy = box_xyxy[cls_keep]
    # conf = conf[cls_keep]
    # class_id = class_id[cls_keep]

    # # 7. Масштабирование bbox'ов обратно в оригинальное изображение (640→orig)
    orig_w, orig_h = pil_img_orig.size
    # box_xyxy сейчас в [0, 640], нужно в [0, orig_w] × [0, orig_h]
    box_xyxy[:, [0, 2]] *= orig_w / imgsz
    box_xyxy[:, [1, 3]] *= orig_h / imgsz

    # Округление до целых
    box_xyxy = box_xyxy.round().int()

    # 8. Отрисовка
    result_img = pil_img_orig.copy()
    draw = ImageDraw.Draw(result_img)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", size=16)
    except:
        font = ImageFont.load_default()

    colors = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (0, 255, 255),
        (255, 0, 255), (128, 0, 0), (0, 128, 0), (0, 0, 128), (128, 128, 0)
    ]

    for i in range(len(box_xyxy)):
        x1, y1, x2, y2 = box_xyxy[i].tolist()
        cls = int(class_id[i].item())
        prob = conf[i].item()
        
        label = f"{class_names[cls]} {prob:.2f}"
        color = colors[cls % len(colors)]
        
        # 1. Рисуем bbox
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)  # ширина 3 → лучше видно
        
        # 2. Подпись — НАД bbox'ом, на контрастной подложке
        # Определяем позицию подписи: сверху-слева от bbox
        text_x = x1
        text_y = max(y1 - 25, 0)  # не вылезаем за верх
        
        # Размер подписи
        try:
            font = ImageFont.truetype("Arial.ttf", 40)
        except:
            try:
                font = ImageFont.truetype("DejaVuSans-Bold.ttf", 40)
            except:
                font = ImageFont.load_default(20)
        
        # Белый жирный текст
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=font, stroke_width=1, stroke_fill=(0, 0, 0))


    return result_img

In [29]:
model = create_yolo_model().to(device).eval()
model.apply(lambda m: hasattr(m, 'reparameterize') and m.reparameterize())
class_names = ['ball', 'coach', 'goalkeeper', 'player', 'referee']  # или dict
model.load_state_dict(torch.load(WEIGHTS_PATH + 'yolo_final_rep_v5.pt'))

# ИЗМЕНИТЬ ДЛЯ ИСПОЛЬЗОВАНИЯ
PROJECT_PATH =  "D:/model_descriptor/ML_MAGA/project/"
GENERAL_PATH = PROJECT_PATH + "coco_yolo/"
TEST_PATH =    PROJECT_PATH + "dataset/test/"
WEIGHTS_PATH = PROJECT_PATH + "model/weights/"

# Тест на изображении
img_path = TEST_PATH + "frame_22959_png.rf.328de5f62c69429926b33e5d1b1761a2.jpg"
result_img = predict_and_visualize_v2(
    model, img_path, class_names,
    device='cuda',
    imgsz=640,
    conf_thres=0.30,   # можно снизить для мелких объектов (мяч!)
    iou_thres=0.1
)

# Показать
result_img.show()